# tanque-cheio · 03 — previsões de um modelo (formato 5.3)

Gera `<system>__<split>.jsonl` em `/kaggle/working/` com uma previsão por pergunta do
dataset (`datasets/dev.jsonl` ou `datasets/test.jsonl` do repositório), no formato da
seção 5.3 do `plan.md`. Usa o prompt único (`tanque.prompting.build_prompt`) pelo chat
template do modelo, decodificação gulosa e `max_new_tokens=256`. A lógica está em
`tanque.infer.local` (testada sem GPU no repositório).

**No Kaggle:** Settings → Accelerator → **GPU T4**; Settings → **Internet on**. Depois
*Run All* (ou *Save Version → Save & Run All* para ter a saída baixável).

As métricas não são calculadas aqui: baixe o JSONL e rode
`uv run tanque eval score datasets/<split>.jsonl results/predictions/<arquivo>` no
computador, onde está o banco.

In [ ]:
# Parâmetros: edite só esta célula.
REPO_REF = "m5-bakeoff"  # branch, tag ou commit do tanque-cheio (depois do merge: "main")
MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"  # repo do modelo no Hugging Face
MODEL_REVISION = None  # commit do checkpoint no HF; None = main (o commit usado vai no run-info)
ADAPTER_ID = None  # adapter LoRA do M6 (repo HF ou pasta); None = modelo base
DATASET = "dev"  # "dev" ou "test"
SYSTEM_NAME = None  # None = base-<modelo> (ou ft-<modelo> com adapter)
LIMIT = None  # ex.: 3 para um teste rápido (o arquivo ganha o sufixo -sample3)
CHAT_TEMPLATE_KWARGS = {}  # Qwen3 / SmolLM3: {"enable_thinking": False}
SYSTEM_SUFFIX = ""  # SmolLM3: "/system_override" (tira a data do dia do template)
DTYPE = "float16"  # a T4 não tem bf16 nativo; "float32" se sair NaN/lixo
DEVICE = "cuda"
MAX_NEW_TOKENS = 256
OUT_DIR = "/kaggle/working"  # vira a saída do notebook no Kaggle
REPO_DIR = "/tmp/tanque-cheio"  # fora de OUT_DIR, para não ir para a saída

In [ ]:
# Instala o pacote `tanque` no REPO_REF e clona o repositório no mesmo ref, para ter
# datasets/, configs/ e data/manifest.json (o prompt usa a data de referência do manifest).
import os
import subprocess
import sys

REPO_URL = "https://github.com/belasoaress/tanque-cheio"


def sh(*cmd: str) -> None:
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)


sh(
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    f"tanque @ git+{REPO_URL}@{REPO_REF}",
    "transformers>=4.53",
    "accelerate",
    "peft",
)
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "origin", REPO_REF)
sh("git", "-C", REPO_DIR, "checkout", "--quiet", "FETCH_HEAD")
REPO_COMMIT = subprocess.check_output(
    ["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True
).strip()
os.environ["TANQUE_ROOT"] = REPO_DIR  # antes de importar `tanque`
print("tanque-cheio em", REPO_COMMIT)

In [ ]:
import torch
import transformers

print("torch", torch.__version__, "| transformers", transformers.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Sem GPU: ligue o acelerador GPU T4 em Settings > Accelerator.")

In [ ]:
import json
from pathlib import Path

from tanque import config
from tanque.infer import local

info = local.run_predictions(
    config.resolve_path(f"datasets/{DATASET}.jsonl"),
    MODEL_ID,
    out_dir=OUT_DIR,
    adapter_id=ADAPTER_ID,
    revision=MODEL_REVISION,
    system=SYSTEM_NAME,
    limit=LIMIT,
    dtype=DTYPE,
    device=DEVICE,
    max_new_tokens=MAX_NEW_TOKENS,
    chat_template_kwargs=CHAT_TEMPLATE_KWARGS,
    system_suffix=SYSTEM_SUFFIX,
)
info.update(repo_ref=REPO_REF, repo_commit=REPO_COMMIT)
run_info = Path(OUT_DIR) / f"run-info__{Path(info['path']).stem}.json"
run_info.write_text(json.dumps(info, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(json.dumps(info, ensure_ascii=False, indent=2))

In [ ]:
# Primeiras previsões, para conferir o formato e a SQL extraída.
for line in Path(info["path"]).read_text(encoding="utf-8").splitlines()[:3]:
    p = json.loads(line)
    print(p["id"], p["latency_ms"], "ms |", p["error"] or p["sql"][:200].replace("\n", " "))